# S³ Spectral-Spatial Domain Adaptation — Binary Left-vs-Right MI (Tuned Final)

This notebook is a clean, Run-All-safe implementation for **binary Left Hand MI vs Right Hand MI** on PhysioNet EEGMMIDB.

### Fixed evaluation targets
`S004, S015, S023, S029, S031, S042, S055, S071, S082, S095`

### Main changes in this version
- Single import/configuration source of truth.
- Cached EDF loading: EDFs are read and epoched once, then reused across folds.
- Correct MNE T1/T2 event-code resolution.
- Exact binary labels: T1→Left (0), T2→Right (1).
- Subject-level validation split using source subjects only.
- Validation-selected checkpoint; target subjects remain unseen.
- Classification-first objective with light SupCon and gradual domain adaptation.
- Conservative augmentation, disabled by default because the previous run suggested transfer degradation.
- Multi-scale temporal encoder + BiGRU + SE attention.
- AdaBN target-statistics adaptation uses unlabeled target EEG only.
- Balanced source sampling.
- Complete binary metrics and paper-ready exports.
- No claims about guaranteed accuracy; the notebook reports the achieved result.

> **Terminology note:** `SimplifiedBiMamba` is retained for continuity, but its implementation is a bidirectional GRU, not a true Mamba SSM.


In [1]:
# ============================================================
# CELL 01 — IMPORTS
# ============================================================

from pathlib import Path
import os
import gc
import json
import math
import random
import time
import warnings
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("[OK] Imports loaded.")


[OK] Imports loaded.


In [2]:
# ============================================================
# CELL 02 — CONFIGURATION + DEVICE
# ============================================================

SEED = 42

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

DATA_DIR = "./eegmmidb"

RUNS = [4, 8, 12]

TMIN = 0.0
TMAX = 4.0
FS = 250.0
N_CHANNELS = 22

# Binary task
N_CLASSES = 2

CLASS_NAMES = [
    "Left Hand MI",
    "Right Hand MI"
]

# ------------------------------------------------------------
# EXACT TARGET SUBJECTS
# ------------------------------------------------------------

TEST_SUBJECTS = [
    4, 15, 23, 29, 31,
    42, 55, 71, 82, 95
]

TOTAL_SUBJECTS = 109
DOMAIN_CLASSES = TOTAL_SUBJECTS

# ------------------------------------------------------------
# Source validation
# ------------------------------------------------------------

VAL_FRACTION = 0.10
VAL_SEED = 123

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

BATCH_SIZE = 64

NUM_EPOCHS = 120

MIN_EPOCHS = 40
PATIENCE = 25

LR = 3e-4
MIN_LR = 1e-6

WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.02

GRAD_CLIP = 1.0

# ------------------------------------------------------------
# Auxiliary objectives
#
# Classification is intentionally dominant.
# ------------------------------------------------------------

SUPCON_WEIGHT = 0.05
SUPCON_TEMP = 0.10

DOMAIN_WEIGHT_MAX = 0.05
DOMAIN_WARMUP_EPOCHS = 30
GRL_START_EPOCH = 15

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

NUM_FILTERS = 10
SINC_KERNEL = 81
SPATIAL_DIM = 64

# ------------------------------------------------------------
# Augmentation
#
# Conservative and OFF by default.
# ------------------------------------------------------------

AUGMENT_PROB = 0.0

NOISE_STD = 0.010
AMPLITUDE_JITTER = 0.03
CHANNEL_DROPOUT = 0.02

# ------------------------------------------------------------
# Optional source-only hyperparameter screening
# ------------------------------------------------------------
#
# Set True for a small validation-only screen.
# It NEVER sees target subjects.
#
# False is recommended for the first full run.
# ------------------------------------------------------------

RUN_SOURCE_TUNING = False

TUNE_EPOCHS = 35

# Candidate tuples:
# (learning_rate, domain_weight, supcon_weight)
TUNE_CANDIDATES = [
    (3e-4, 0.00, 0.05),
    (3e-4, 0.03, 0.05),
    (2e-4, 0.05, 0.05),
    (3e-4, 0.03, 0.00),
]

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

RESULTS_DIR = Path(
    "./results_binary_left_right_tuned_final"
)

FIG_DIR = RESULTS_DIR / "figures"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

def seed_everything(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

seed_everything()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)

print("Task       :", "Left Hand MI vs Right Hand MI")
print("Runs       :", RUNS)
print("Targets    :", [f"S{s:03d}" for s in TEST_SUBJECTS])
print("Device     :", DEVICE)
print("Epochs     :", NUM_EPOCHS)
print("Batch size :", BATCH_SIZE)
print("LR         :", LR)
print("Domain max :", DOMAIN_WEIGHT_MAX)
print("SupCon     :", SUPCON_WEIGHT)
print("Augment    :", AUGMENT_PROB)
print("Source tune:", RUN_SOURCE_TUNING)
print("Results    :", RESULTS_DIR.resolve())


CONFIGURATION
Task       : Left Hand MI vs Right Hand MI
Runs       : [4, 8, 12]
Targets    : ['S004', 'S015', 'S023', 'S029', 'S031', 'S042', 'S055', 'S071', 'S082', 'S095']
Device     : mps
Epochs     : 120
Batch size : 64
LR         : 0.0003
Domain max : 0.05
SupCon     : 0.05
Augment    : 0.0
Source tune: False
Results    : /Users/ashokvarmabevara/Project2/results_binary_left_right_tuned_final


In [3]:
# ============================================================
# CELL 03 — PREFLIGHT / DATASET CHECK
# ============================================================

def discover_available_subjects(
    data_dir,
    max_subjects=TOTAL_SUBJECTS
):
    root = Path(data_dir)

    return [
        s for s in range(1, max_subjects + 1)
        if (root / f"S{s:03d}").is_dir()
    ]


available_subjects = discover_available_subjects(
    DATA_DIR
)

print(
    "Available subject folders:",
    len(available_subjects)
)

missing_targets = [
    s for s in TEST_SUBJECTS
    if s not in available_subjects
]

if missing_targets:
    raise FileNotFoundError(
        "Missing target subjects: "
        +
        ", ".join(
            f"S{s:03d}"
            for s in missing_targets
        )
    )

missing_target_files = []

for subject in TEST_SUBJECTS:

    folder = Path(DATA_DIR) / f"S{subject:03d}"

    for run in RUNS:

        path = (
            folder /
            f"S{subject:03d}R{run:02d}.edf"
        )

        if not path.exists():
            missing_target_files.append(str(path))

if missing_target_files:

    print("[WARNING] Missing target EDFs:")

    for path in missing_target_files:
        print(" ", path)

else:

    print(
        "[OK] All requested target EDF files exist."
    )

print(
    "[OK] Preflight complete."
)


Available subject folders: 109
[OK] All requested target EDF files exist.
[OK] Preflight complete.


## CELL 04 — Cached binary EEGMMIDB loader

This loader resolves the **actual MNE event codes** for T1/T2 instead of assuming event ID `0/1`. It loads only runs 4, 8, and 12 and maps:

- T1 → class 0 → Left Hand MI
- T2 → class 1 → Right Hand MI

Each trial is normalized independently per channel before it enters the model.


In [4]:
# ============================================================
# CELL 04 — CACHED BINARY EEG LOADER
# ============================================================

class EEGMMIDB_Cache:

    def __init__(
        self,
        data_dir,
        subjects,
        runs=RUNS,
        tmin=TMIN,
        tmax=TMAX,
        fs=FS,
        n_channels=N_CHANNELS
    ):

        self.data_dir = str(data_dir)
        self.subjects = list(subjects)

        self.runs = list(runs)

        self.tmin = tmin
        self.tmax = tmax
        self.fs = fs
        self.n_channels = n_channels

        samples = []
        labels = []
        subject_ids = []
        subject_numbers = []
        run_ids = []

        start = time.time()

        loaded_runs = 0
        loaded_trials = 0

        # ----------------------------------------------------
        # Event-code resolver
        # ----------------------------------------------------

        @staticmethod
        def _clean(name):
            return str(name).strip().upper()

        for subject in self.subjects:

            subject_folder = (
                Path(self.data_dir) /
                f"S{subject:03d}"
            )

            if not subject_folder.is_dir():
                continue

            for run in self.runs:

                edf = (
                    subject_folder /
                    f"S{subject:03d}R{run:02d}.edf"
                )

                if not edf.exists():
                    continue

                try:

                    # ------------------------------------------
                    # Load EDF
                    # ------------------------------------------

                    raw = mne.io.read_raw_edf(
                        str(edf),
                        preload=True,
                        verbose=False
                    )

                    if (
                        len(raw.ch_names)
                        <
                        self.n_channels
                    ):

                        print(
                            f"[WARN] S{subject:03d} "
                            f"R{run:02d}: fewer than "
                            f"{self.n_channels} channels."
                        )

                        continue

                    # ------------------------------------------
                    # First 22 channels
                    # ------------------------------------------

                    raw.pick(
                        raw.ch_names[:self.n_channels]
                    )

                    # ------------------------------------------
                    # Resample
                    # ------------------------------------------

                    raw.resample(
                        self.fs,
                        npad="auto"
                    )

                    # ------------------------------------------
                    # Annotation events
                    # ------------------------------------------

                    events, event_id = (
                        mne.events_from_annotations(
                            raw,
                            verbose=False
                        )
                    )

                    # ------------------------------------------
                    # Robustly find T1/T2
                    # ------------------------------------------

                    t1_code = None
                    t2_code = None

                    for description, code in event_id.items():

                        desc = self._clean(
                            description
                        )

                        if (
                            desc == "T1"
                            or
                            desc.startswith("T1/")
                            or
                            desc.startswith("T1-")
                            or
                            desc.startswith("T1_")
                        ):
                            t1_code = int(code)

                        if (
                            desc == "T2"
                            or
                            desc.startswith("T2/")
                            or
                            desc.startswith("T2-")
                            or
                            desc.startswith("T2_")
                        ):
                            t2_code = int(code)

                    if (
                        t1_code is None
                        or
                        t2_code is None
                    ):

                        print(
                            f"[WARN] S{subject:03d} "
                            f"R{run:02d}: T1/T2 not found. "
                            f"Available={list(event_id.keys())}"
                        )

                        continue

                    # ------------------------------------------
                    # Only Left/Right class runs
                    # ------------------------------------------

                    epochs = mne.Epochs(
                        raw,
                        events,
                        event_id={
                            "T1": t1_code,
                            "T2": t2_code
                        },
                        tmin=self.tmin,
                        tmax=self.tmax - 1.0 / self.fs,
                        baseline=None,
                        preload=True,
                        reject_by_annotation=True,
                        verbose=False
                    )

                    if len(epochs) == 0:
                        continue

                    data = epochs.get_data(
                        copy=True
                    )

                    codes = epochs.events[:, -1]

                    trial_count = 0

                    # ------------------------------------------
                    # Convert MNE codes to binary labels
                    # ------------------------------------------

                    for i in range(len(data)):

                        actual_code = int(
                            codes[i]
                        )

                        if actual_code == t1_code:

                            label = 0

                        elif actual_code == t2_code:

                            label = 1

                        else:

                            continue

                        trial = (
                            data[i]
                            .astype(np.float32)
                        )

                        # --------------------------------------
                        # Per-trial / per-channel Z-score
                        # --------------------------------------

                        mean = trial.mean(
                            axis=1,
                            keepdims=True
                        )

                        std = trial.std(
                            axis=1,
                            keepdims=True
                        )

                        trial = (
                            trial - mean
                        ) / (
                            std + 1e-6
                        )

                        samples.append(trial)

                        labels.append(label)

                        subject_ids.append(
                            subject - 1
                        )

                        subject_numbers.append(
                            subject
                        )

                        run_ids.append(
                            run
                        )

                        trial_count += 1

                    loaded_runs += 1
                    loaded_trials += trial_count

                    print(
                        f"[OK] S{subject:03d} "
                        f"R{run:02d} | "
                        f"T1={t1_code}->Left | "
                        f"T2={t2_code}->Right | "
                        f"epochs={trial_count}"
                    )

                except Exception as exc:

                    print(
                        f"[WARN] S{subject:03d} "
                        f"R{run:02d}: "
                        f"{type(exc).__name__}: {exc}"
                    )

        # ----------------------------------------------------
        # Build arrays
        # ----------------------------------------------------

        n_time = int(
            self.fs
            *
            (self.tmax - self.tmin)
        )

        if samples:

            self.X = np.stack(
                samples
            ).astype(
                np.float32
            )

            self.y = np.asarray(
                labels,
                dtype=np.int64
            )

            self.subject_id = np.asarray(
                subject_ids,
                dtype=np.int64
            )

            self.subject_number = np.asarray(
                subject_numbers,
                dtype=np.int64
            )

            self.run_id = np.asarray(
                run_ids,
                dtype=np.int64
            )

        else:

            self.X = np.empty(
                (
                    0,
                    self.n_channels,
                    n_time
                ),
                dtype=np.float32
            )

            self.y = np.empty(
                (0,),
                dtype=np.int64
            )

            self.subject_id = np.empty(
                (0,),
                dtype=np.int64
            )

            self.subject_number = np.empty(
                (0,),
                dtype=np.int64
            )

            self.run_id = np.empty(
                (0,),
                dtype=np.int64
            )

        print()
        print(
            "=" * 70
        )

        print(
            f"Dataset loading complete: "
            f"{loaded_trials} binary trials"
        )

        print(
            "Runs loaded:",
            loaded_runs
        )

        print(
            "X shape:",
            self.X.shape
        )

        print(
            "RAM:",
            f"{self.X.nbytes / 1024**3:.3f} GB"
        )

        print(
            "Time:",
            f"{(time.time() - start) / 60:.2f} min"
        )

        print(
            "=" * 70
        )

    def subset(self, subjects):

        wanted = np.asarray(
            list(subjects),
            dtype=np.int64
        )

        mask = np.isin(
            self.subject_number,
            wanted
        )

        indices = np.flatnonzero(
            mask
        )

        return EEGMMIDB_Subset(
            self,
            indices
        )


class EEGMMIDB_Subset(Dataset):

    def __init__(
        self,
        cache,
        indices
    ):

        self.cache = cache

        self.indices = np.asarray(
            indices,
            dtype=np.int64
        )

        self.labels = (
            cache.y[
                self.indices
            ]
        )

        self.subjects = (
            cache.subject_number[
                self.indices
            ]
        )

    def __len__(self):

        return len(
            self.indices
        )

    def __getitem__(self, idx):

        j = int(
            self.indices[idx]
        )

        x = torch.from_numpy(
            self.cache.X[j]
        )

        y = torch.tensor(
            int(self.cache.y[j]),
            dtype=torch.long
        )

        subject = torch.tensor(
            int(self.cache.subject_id[j]),
            dtype=torch.long
        )

        return (
            x,
            y,
            subject
        )


# ------------------------------------------------------------
# Load the complete binary cache ONCE
# ------------------------------------------------------------

CACHE_SUBJECTS = available_subjects

EEG_CACHE = EEGMMIDB_Cache(
    DATA_DIR,
    CACHE_SUBJECTS
)

print()
print(
    "Binary classes:",
    {
        int(c): int((EEG_CACHE.y == c).sum())
        for c in np.unique(EEG_CACHE.y)
    }
)


[WARN] S001 R04: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S001 R08: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S001 R12: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S002 R04: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S002 R08: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S002 R12: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S003 R04: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S003 R08: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S003 R12: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S004 R04: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S004 R08: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WARN] S004 R12: AttributeError: 'EEGMMIDB_Cache' object has no attribute '_clean'
[WAR

In [5]:
# ============================================================
# CELL 05 — TARGET DATA SANITY CHECK
# ============================================================

print("=" * 80)
print("TARGET SUBJECT SANITY CHECK")
print("=" * 80)

target_rows = []

for subject in TEST_SUBJECTS:

    ds = EEG_CACHE.subset(
        [subject]
    )

    labels = np.asarray(
        ds.labels
    )

    counts = {
        "Left": int(
            (labels == 0).sum()
        ),
        "Right": int(
            (labels == 1).sum()
        )
    }

    print(
        f"S{subject:03d} | "
        f"Total={len(ds)} | "
        f"Left={counts['Left']} | "
        f"Right={counts['Right']}"
    )

    if (
        counts["Left"] == 0
        or
        counts["Right"] == 0
    ):

        raise RuntimeError(
            f"S{subject:03d} does not contain "
            "both Left and Right classes."
        )

    target_rows.append({
        "subject": subject,
        "total": len(ds),
        "left": counts["Left"],
        "right": counts["Right"]
    })

target_df = pd.DataFrame(
    target_rows
)

print()
print(
    target_df.to_string(
        index=False
    )
)

print()
print("[OK] All target subjects contain both classes.")


TARGET SUBJECT SANITY CHECK
S004 | Total=0 | Left=0 | Right=0


RuntimeError: S004 does not contain both Left and Right classes.

In [ ]:
# ============================================================
# CELL 06 — GRADIENT REVERSAL + SINC FILTER BANK
# ============================================================

class GradientReversalLayer(
    torch.autograd.Function
):

    @staticmethod
    def forward(
        ctx,
        x,
        lambda_grl
    ):

        ctx.lambda_grl = float(
            lambda_grl
        )

        return x.view_as(x)

    @staticmethod
    def backward(
        ctx,
        grad_output
    ):

        return (
            -ctx.lambda_grl
            *
            grad_output,
            None
        )


def grl(
    x,
    lambda_grl=1.0
):

    return GradientReversalLayer.apply(
        x,
        lambda_grl
    )


class SincFilterBank(nn.Module):

    def __init__(
        self,
        in_channels=N_CHANNELS,
        num_filters=NUM_FILTERS,
        kernel_size=SINC_KERNEL,
        sample_rate=FS
    ):

        super().__init__()

        if kernel_size % 2 == 0:
            raise ValueError(
                "SINC_KERNEL must be odd."
            )

        self.in_channels = (
            in_channels
        )

        self.num_filters = (
            num_filters
        )

        self.kernel_size = (
            kernel_size
        )

        self.sample_rate = (
            sample_rate
        )

        # Stable learnable parameterization
        self.low_raw = nn.Parameter(
            torch.rand(
                num_filters
            )
            *
            8.0
            +
            4.0
        )

        self.band_raw = nn.Parameter(
            torch.rand(
                num_filters
            )
            *
            12.0
            +
            8.0
        )

    def forward(self, x):

        B, C, T = x.shape

        if C != self.in_channels:

            raise ValueError(
                f"Expected {self.in_channels} channels, "
                f"got {C}"
            )

        n = torch.arange(
            -(self.kernel_size // 2),
            self.kernel_size // 2 + 1,
            device=x.device,
            dtype=x.dtype
        )

        nyquist = (
            self.sample_rate / 2.0
        )

        filters = []

        for i in range(
            self.num_filters
        ):

            low = (
                F.softplus(
                    self.low_raw[i]
                )
                +
                1.0
            ).clamp(
                1.0,
                nyquist - 6.0
            )

            high = (
                low
                +
                F.softplus(
                    self.band_raw[i]
                )
                +
                2.0
            ).clamp(
                3.0,
                nyquist - 1.0
            )

            high = torch.maximum(
                high,
                low + 1.0
            ).clamp(
                max=nyquist - 1.0
            )

            f_low = (
                low
                /
                self.sample_rate
            )

            f_high = (
                high
                /
                self.sample_rate
            )

            # Sinc band-pass
            w = (
                2
                *
                f_high
                *
                torch.sinc(
                    2
                    *
                    f_high
                    *
                    n
                )
                -
                2
                *
                f_low
                *
                torch.sinc(
                    2
                    *
                    f_low
                    *
                    n
                )
            )

            # Hamming window
            window = torch.hamming_window(
                self.kernel_size,
                periodic=False,
                device=x.device,
                dtype=x.dtype
            )

            w = w * window

            # Normalize
            w = (
                w
                /
                (
                    torch.sum(
                        torch.abs(w)
                    )
                    +
                    1e-6
                )
            )

            filters.append(
                w.view(1, 1, -1)
            )

        filters = torch.cat(
            filters,
            dim=0
        )

        # One filter shared across all EEG channels
        x_reshape = x.reshape(
            B * C,
            1,
            T
        )

        out = F.conv1d(
            x_reshape,
            filters,
            padding="same"
        )

        return (
            out
            .reshape(
                B,
                C,
                self.num_filters,
                T
            )
            .permute(
                0,
                2,
                1,
                3
            )
        )


print(
    "[OK] GRL and SincFilterBank defined."
)


In [ ]:
# ============================================================
# CELL 07 — DYNAMIC GRAPH NEURAL NETWORK
# ============================================================

class DGNN(nn.Module):

    def __init__(
        self,
        num_filters=NUM_FILTERS,
        in_nodes=N_CHANNELS,
        out_nodes=SPATIAL_DIM
    ):

        super().__init__()

        self.num_filters = (
            num_filters
        )

        self.in_nodes = (
            in_nodes
        )

        self.out_nodes = (
            out_nodes
        )

        self.W_Q = nn.Linear(
            num_filters,
            num_filters
        )

        self.W_K = nn.Linear(
            num_filters,
            num_filters
        )

        self.W_V = nn.Linear(
            in_nodes,
            out_nodes
        )

        self.norm = nn.LayerNorm(
            out_nodes
        )

    def forward(self, x):

        # x = [B, F, C, T]

        B, F_band, C, T = x.shape

        if F_band != self.num_filters:
            raise ValueError(
                f"Expected {self.num_filters} bands, "
                f"got {F_band}"
            )

        if C != self.in_nodes:
            raise ValueError(
                f"Expected {self.in_nodes} channels, "
                f"got {C}"
            )

        # ----------------------------------------------------
        # Channel descriptors
        # [B,F,C,T] -> [B,C,F]
        # ----------------------------------------------------

        descriptor = (
            x.mean(dim=-1)
            .transpose(1, 2)
        )

        Q = self.W_Q(
            descriptor
        )

        K = self.W_K(
            descriptor
        )

        # ----------------------------------------------------
        # Dynamic adjacency
        # ----------------------------------------------------

        A = (
            torch.matmul(
                Q,
                K.transpose(
                    -2,
                    -1
                )
            )
            /
            math.sqrt(
                self.num_filters
            )
        )

        A = F.softmax(
            A,
            dim=-1
        )

        # Self loops
        eye = torch.eye(
            C,
            device=x.device,
            dtype=x.dtype
        ).unsqueeze(0)

        A = A + eye

        # Symmetric normalization
        degree = A.sum(
            dim=-1
        ).clamp_min(
            1e-6
        )

        D = torch.diag_embed(
            torch.rsqrt(
                degree
            )
        )

        A_norm = (
            D
            @
            A
            @
            D
        )

        # ----------------------------------------------------
        # Propagate graph
        # ----------------------------------------------------

        x_t = x.permute(
            0,
            1,
            3,
            2
        )
        # [B,F,T,C]

        out = torch.einsum(
            "bij,bftj->bfti",
            A_norm,
            x_t
        )
        # [B,F,T,C]

        out = self.W_V(
            out
        )
        # [B,F,T,D]

        out = F.elu(
            out
        )

        out = out.permute(
            0,
            1,
            3,
            2
        )
        # [B,F,D,T]

        return self.norm(
            out.transpose(
                -1,
                -2
            )
        ).transpose(
            -1,
            -2
        )


print("[OK] DGNN defined.")


In [ ]:
# ============================================================
# CELL 08 — MULTI-SCALE TEMPORAL ENCODER + BIGRU + SE
# ============================================================

class MultiScaleTemporalEncoder(
    nn.Module
):

    def __init__(
        self,
        channels=SPATIAL_DIM
    ):

        super().__init__()

        # ----------------------------------------------------
        # 3 temporal scales
        # ----------------------------------------------------

        self.branch3 = nn.Sequential(
            nn.Conv1d(
                channels,
                channels,
                kernel_size=3,
                padding=1,
                groups=channels,
                bias=False
            ),
            nn.BatchNorm1d(
                channels
            ),
            nn.ELU(
                inplace=True
            )
        )

        self.branch7 = nn.Sequential(
            nn.Conv1d(
                channels,
                channels,
                kernel_size=7,
                padding=3,
                groups=channels,
                bias=False
            ),
            nn.BatchNorm1d(
                channels
            ),
            nn.ELU(
                inplace=True
            )
        )

        self.branch15 = nn.Sequential(
            nn.Conv1d(
                channels,
                channels,
                kernel_size=15,
                padding=7,
                groups=channels,
                bias=False
            ),
            nn.BatchNorm1d(
                channels
            ),
            nn.ELU(
                inplace=True
            )
        )

        # ----------------------------------------------------
        # Fuse
        # ----------------------------------------------------

        self.fuse = nn.Sequential(
            nn.Conv1d(
                channels * 3,
                channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm1d(
                channels
            ),
            nn.ELU(
                inplace=True
            ),

            # 1000 -> 250
            nn.AvgPool1d(
                kernel_size=4,
                stride=4
            )
        )

    def forward(self, x):

        a = self.branch3(x)
        b = self.branch7(x)
        c = self.branch15(x)

        y = torch.cat(
            [a, b, c],
            dim=1
        )

        return self.fuse(y)


class SimplifiedBiMamba(
    nn.Module
):

    """
    Compatibility name from the supplied code.

    Actual implementation:
        Bidirectional GRU
    """

    def __init__(
        self,
        d_model=SPATIAL_DIM
    ):

        super().__init__()

        self.ssm = nn.GRU(
            input_size=d_model,
            hidden_size=d_model // 2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):

        y, _ = self.ssm(
            x.transpose(
                1,
                2
            )
        )

        return y.transpose(
            1,
            2
        )


class SEAttention(nn.Module):

    def __init__(
        self,
        channel=SPATIAL_DIM,
        reduction=16
    ):

        super().__init__()

        hidden = max(
            1,
            channel // reduction
        )

        self.fc = nn.Sequential(

            nn.Linear(
                channel,
                hidden,
                bias=False
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                hidden,
                channel,
                bias=False
            ),

            nn.Sigmoid()
        )

    def forward(self, x):

        # x = [B,C,T]

        b, c, t = x.size()

        pooled = x.mean(
            dim=2
        )

        weights = self.fc(
            pooled
        ).view(
            b,
            c,
            1
        )

        weighted = (
            x
            *
            weights
        )

        return weighted.mean(
            dim=2
        )


print(
    "[OK] Temporal, BiGRU and SE modules defined."
)


In [ ]:
# ============================================================
# CELL 09 — COMPLETE BINARY S³ MODEL
# ============================================================

class S3MambaDA(nn.Module):

    def __init__(
        self,
        num_classes=N_CLASSES,
        num_subjects=DOMAIN_CLASSES
    ):

        super().__init__()

        # Spectral
        self.sinc_filter = SincFilterBank()

        # Spatial
        self.dgnn = DGNN()

        # Temporal
        self.temporal = (
            MultiScaleTemporalEncoder()
        )

        self.mamba = (
            SimplifiedBiMamba()
        )

        # Attention
        self.se_attention = (
            SEAttention()
        )

        # ----------------------------------------------------
        # Main binary classifier
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.BatchNorm1d(
                SPATIAL_DIM
            ),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                SPATIAL_DIM,
                num_classes
            )
        )

        # ----------------------------------------------------
        # Domain classifier
        # ----------------------------------------------------

        self.domain_classifier = nn.Sequential(

            nn.Linear(
                SPATIAL_DIM,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                0.10
            ),

            nn.Linear(
                64,
                num_subjects
            )
        )

        # ----------------------------------------------------
        # SupCon projection
        # ----------------------------------------------------

        self.supcon_proj = nn.Sequential(

            nn.Linear(
                SPATIAL_DIM,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                128,
                128
            )
        )

    def forward(
        self,
        x,
        lambda_grl=0.0
    ):

        # [B,22,T]
        x = self.sinc_filter(x)

        # [B,10,22,T]
        x = self.dgnn(x)

        # Average learned bands
        x = x.mean(
            dim=1
        )

        # [B,64,T]
        x = self.temporal(x)

        # [B,64,250]
        x = self.mamba(x)

        # [B,64]
        z = self.se_attention(x)

        # Classification
        class_logits = self.classifier(z)

        # Domain
        z_domain = grl(
            z,
            lambda_grl
        )

        domain_logits = (
            self.domain_classifier(
                z_domain
            )
        )

        # SupCon
        z_proj = F.normalize(
            self.supcon_proj(z),
            p=2,
            dim=1
        )

        return (
            class_logits,
            domain_logits,
            z_proj,
            z
        )


print("[OK] S3MambaDA defined.")


In [ ]:
# ============================================================
# CELL 10 — SUPCON + ADABN
# ============================================================

class SupConLoss(
    nn.Module
):

    def __init__(
        self,
        temperature=SUPCON_TEMP
    ):

        super().__init__()

        self.temperature = (
            temperature
        )

    def forward(
        self,
        features,
        labels
    ):

        device = features.device

        n = features.shape[0]

        if n < 2:
            return features.sum() * 0.0

        sim = (
            torch.matmul(
                features,
                features.T
            )
            /
            self.temperature
        )

        labels = labels.view(
            -1,
            1
        )

        mask = (
            torch.eq(
                labels,
                labels.T
            )
            .float()
        )

        logits_mask = torch.ones_like(
            mask
        )

        logits_mask.fill_diagonal_(
            0
        )

        mask = (
            mask
            *
            logits_mask
        )

        # Stable denominator
        logits = (
            sim
            -
            sim.max(
                dim=1,
                keepdim=True
            ).values.detach()
        )

        exp_logits = (
            torch.exp(
                logits
            )
            *
            logits_mask
        )

        log_prob = (
            logits
            -
            torch.log(
                exp_logits.sum(
                    dim=1,
                    keepdim=True
                )
                +
                1e-8
            )
        )

        positives = mask.sum(
            dim=1
        )

        valid = (
            positives
            >
            0
        )

        if not valid.any():

            return features.sum() * 0.0

        mean_pos = (
            mask
            *
            log_prob
        ).sum(
            dim=1
        ) / (
            positives
            +
            1e-8
        )

        return -mean_pos[
            valid
        ].mean()


@torch.no_grad()
def apply_adabn(
    model,
    loader,
    device,
    adaptation_trials=20
):

    # --------------------------------------------------------
    # Collect unlabeled target EEG
    # --------------------------------------------------------

    batches = []
    seen = 0

    for x, _, _ in loader:

        batches.append(
            x
        )

        seen += x.size(0)

        if seen >= adaptation_trials:
            break

    if not batches:
        return model

    target_x = (
        torch.cat(
            batches,
            dim=0
        )[:adaptation_trials]
        .to(device)
    )

    # --------------------------------------------------------
    # Find BatchNorm modules
    # --------------------------------------------------------

    bn_modules = [
        m
        for m in model.modules()
        if isinstance(
            m,
            nn.modules.batchnorm._BatchNorm
        )
    ]

    if not bn_modules:
        return model

    saved = []

    for m in bn_modules:

        saved.append(
            (
                m.training,
                m.momentum,
                m.track_running_stats
            )
        )

        m.reset_running_stats()

        m.momentum = 1.0

        m.train()

    model.eval()

    # Keep BN modules in train mode while all other modules stay eval
    for m in bn_modules:
        m.train()

    # One unlabeled forward pass updates BN statistics
    _ = model(
        target_x,
        lambda_grl=0.0
    )

    # Restore standard evaluation mode
    for m, (
        was_training,
        old_momentum,
        track_stats
    ) in zip(
        bn_modules,
        saved
    ):

        m.momentum = old_momentum

        m.track_running_stats = track_stats

        m.eval()

    model.eval()

    return model


print("[OK] SupCon and AdaBN defined.")


In [ ]:
# ============================================================
# CELL 11 — COMPLETE MODEL SMOKE TEST
# ============================================================

seed_everything(SEED)

model_test = S3MambaDA().to(
    DEVICE
)

x_test = torch.randn(
    2,
    N_CHANNELS,
    int(FS * TMAX),
    device=DEVICE
)

with torch.no_grad():

    (
        cls_test,
        dom_test,
        proj_test,
        emb_test
    ) = model_test(
        x_test,
        lambda_grl=0.0
    )

print("=" * 80)
print("SMOKE TEST")
print("=" * 80)

print(
    "Input         :",
    tuple(x_test.shape)
)

print(
    "Class logits  :",
    tuple(cls_test.shape)
)

print(
    "Domain logits :",
    tuple(dom_test.shape)
)

print(
    "Projection    :",
    tuple(proj_test.shape)
)

print(
    "Embedding     :",
    tuple(emb_test.shape)
)

print(
    "Parameters    :",
    f"{sum(p.numel() for p in model_test.parameters()):,}"
)

assert cls_test.shape == (
    2,
    2
)

assert dom_test.shape == (
    2,
    DOMAIN_CLASSES
)

assert proj_test.shape == (
    2,
    128
)

assert emb_test.shape == (
    2,
    SPATIAL_DIM
)

print()
print(
    "[OK] Model smoke test passed."
)

del model_test
del x_test
del cls_test
del dom_test
del proj_test
del emb_test

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# CELL 12 — SOURCE VALIDATION SPLIT + BALANCED SAMPLING
# ============================================================

def make_subject_validation_split(
    subjects,
    val_fraction=VAL_FRACTION,
    seed=VAL_SEED
):

    subjects = sorted(
        list(subjects)
    )

    rng = np.random.RandomState(
        seed
    )

    shuffled = subjects.copy()

    rng.shuffle(
        shuffled
    )

    n_val = max(
        1,
        int(
            round(
                len(shuffled)
                *
                val_fraction
            )
        )
    )

    val_subjects = sorted(
        shuffled[:n_val]
    )

    train_subjects = sorted(
        shuffled[n_val:]
    )

    return (
        train_subjects,
        val_subjects
    )


def make_balanced_sampler(
    dataset
):

    labels = np.asarray(
        dataset.labels,
        dtype=np.int64
    )

    classes, counts = np.unique(
        labels,
        return_counts=True
    )

    if len(classes) != 2:

        raise RuntimeError(
            "Expected both binary classes "
            "in training data."
        )

    class_weights = {
        int(c): 1.0 / float(n)
        for c, n
        in zip(
            classes,
            counts
        )
    }

    weights = np.asarray(
        [
            class_weights[
                int(y)
            ]
            for y in labels
        ],
        dtype=np.float64
    )

    return WeightedRandomSampler(
        torch.tensor(
            weights,
            dtype=torch.double
        ),
        num_samples=len(weights),
        replacement=True
    )


print(
    "[OK] Validation split and sampler ready."
)


In [ ]:
# ============================================================
# CELL 13 — CONSERVATIVE EEG AUGMENTATION
# ============================================================

def augment_eeg(x):

    if AUGMENT_PROB <= 0.0:
        return x

    if np.random.rand() > AUGMENT_PROB:
        return x

    x = x.clone()

    # Small amplitude jitter
    scale = (
        1.0
        +
        AMPLITUDE_JITTER
        *
        torch.randn(
            x.size(0),
            1,
            1,
            device=x.device
        )
    )

    x = x * scale

    # Tiny Gaussian noise
    x = (
        x
        +
        NOISE_STD
        *
        torch.randn_like(x)
    )

    # Very small channel dropout
    keep = (
        torch.rand(
            x.size(0),
            x.size(1),
            1,
            device=x.device
        )
        >
        CHANNEL_DROPOUT
    )

    x = x * keep

    return x


print(
    "[OK] Augmentation ready. "
    f"Probability={AUGMENT_PROB}"
)


In [ ]:
# ============================================================
# CELL 14 — DOMAIN / GRL SCHEDULE
# ============================================================

def get_domain_weight(
    epoch,
    total_epochs,
    max_weight=DOMAIN_WEIGHT_MAX,
    warmup=DOMAIN_WARMUP_EPOCHS
):

    if epoch < warmup:
        return 0.0

    p = (
        epoch - warmup
    ) / max(
        1,
        total_epochs - warmup
    )

    p = float(
        np.clip(
            p,
            0.0,
            1.0
        )
    )

    # Smooth ramp
    return float(
        max_weight
        *
        0.5
        *
        (
            1.0
            -
            np.cos(
                np.pi * p
            )
        )
    )


def get_grl_lambda(
    epoch,
    total_epochs,
    start_epoch=GRL_START_EPOCH
):

    if epoch < start_epoch:
        return 0.0

    p = (
        epoch - start_epoch
    ) / max(
        1,
        total_epochs - start_epoch
    )

    p = float(
        np.clip(
            p,
            0.0,
            1.0
        )
    )

    return float(
        2.0
        /
        (
            1.0
            +
            np.exp(
                -10.0 * p
            )
        )
        -
        1.0
    )


for e in [
    0,
    9,
    14,
    29,
    59,
    119
]:

    print(
        f"Epoch {e+1:03d} | "
        f"DomainWeight="
        f"{get_domain_weight(e, NUM_EPOCHS):.4f} | "
        f"GRL="
        f"{get_grl_lambda(e, NUM_EPOCHS):.4f}"
    )

print(
    "\n[OK] Domain schedule ready."
)


In [ ]:
# ============================================================
# CELL 15 — EVALUATION HELPER
# ============================================================

@torch.no_grad()
def evaluate_binary(
    model,
    loader,
    device
):

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []
    embeddings = []

    for x, y, _ in loader:

        x = x.to(device)

        (
            logits,
            _,
            _,
            z
        ) = model(
            x,
            lambda_grl=0.0
        )

        prob_right = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        pred = (
            prob_right >= 0.5
        ).long()

        y_true.extend(
            y.numpy().tolist()
        )

        y_pred.extend(
            pred.cpu()
            .numpy()
            .tolist()
        )

        y_prob.extend(
            prob_right.cpu()
            .numpy()
            .tolist()
        )

        embeddings.append(
            z.cpu()
            .numpy()
        )

    y_true = np.asarray(
        y_true,
        dtype=np.int64
    )

    y_pred = np.asarray(
        y_pred,
        dtype=np.int64
    )

    y_prob = np.asarray(
        y_prob,
        dtype=np.float64
    )

    if embeddings:

        embeddings = np.concatenate(
            embeddings,
            axis=0
        )

    else:

        embeddings = np.empty(
            (
                0,
                SPATIAL_DIM
            ),
            dtype=np.float32
        )

    metrics = {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "kappa":
            cohen_kappa_score(
                y_true,
                y_pred
            ),

        "precision_macro":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "recall_macro":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "f1_macro":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "y_prob":
            y_prob,

        "embeddings":
            embeddings
    }

    try:

        metrics["roc_auc"] = (
            roc_auc_score(
                y_true,
                y_prob
            )
        )

    except Exception:

        metrics["roc_auc"] = np.nan

    return metrics


def validation_selection_score(
    metrics
):

    auc_value = (
        0.5
        if np.isnan(
            metrics["roc_auc"]
        )
        else metrics["roc_auc"]
    )

    return float(
        0.60
        *
        metrics["balanced_accuracy"]
        +
        0.25
        *
        auc_value
        +
        0.15
        *
        metrics["kappa"]
    )


print(
    "[OK] Evaluation helper ready."
)


In [ ]:
# ============================================================
# CELL 16 — TRAINING FUNCTION
# ============================================================

def train_model(
    train_ds,
    val_ds,
    fold_idx,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    domain_weight_max=DOMAIN_WEIGHT_MAX,
    supcon_weight=SUPCON_WEIGHT
):

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=make_balanced_sampler(
            train_ds
        ),
        num_workers=0,
        drop_last=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    seed_everything(
        SEED + fold_idx
    )

    model = S3MambaDA().to(
        DEVICE
    )

    criterion_cls = (
        nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING
        )
    )

    criterion_domain = (
        nn.CrossEntropyLoss()
    )

    criterion_supcon = (
        SupConLoss(
            SUPCON_TEMP
        )
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )

    # --------------------------------------------------------
    # Warmup + cosine schedule
    # --------------------------------------------------------

    warmup_epochs = 8

    def lr_factor(epoch):

        if epoch < warmup_epochs:

            return (
                0.20
                +
                0.80
                *
                (
                    epoch + 1
                )
                /
                warmup_epochs
            )

        p = (
            epoch - warmup_epochs
        ) / max(
            1,
            num_epochs - warmup_epochs
        )

        p = float(
            np.clip(
                p,
                0.0,
                1.0
            )
        )

        return (
            MIN_LR / lr
            +
            0.5
            *
            (
                1.0
                -
                MIN_LR / lr
            )
            *
            (
                1.0
                +
                np.cos(
                    np.pi * p
                )
            )
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_factor
    )

    use_amp = (
        DEVICE.type == "cuda"
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )

    best_score = -np.inf

    best_state = None

    best_epoch = 0

    patience_count = 0

    history = []

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(
        num_epochs
    ):

        model.train()

        train_loss = 0.0
        cls_loss_sum = 0.0
        dom_loss_sum = 0.0
        con_loss_sum = 0.0

        seen = 0

        domain_weight = get_domain_weight(
            epoch,
            num_epochs,
            max_weight=domain_weight_max
        )

        grl_lambda = get_grl_lambda(
            epoch,
            num_epochs
        )

        for x, y, subject in train_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)
            subject = subject.to(DEVICE)

            x_input = augment_eeg(
                x
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=use_amp
            ):

                (
                    logits,
                    domain_logits,
                    z_proj,
                    _
                ) = model(
                    x_input,
                    lambda_grl=grl_lambda
                )

                loss_cls = criterion_cls(
                    logits,
                    y
                )

                loss_domain = criterion_domain(
                    domain_logits,
                    subject
                )

                loss_supcon = criterion_supcon(
                    z_proj,
                    y
                )

                loss = (
                    loss_cls
                    +
                    domain_weight
                    *
                    loss_domain
                    +
                    supcon_weight
                    *
                    loss_supcon
                )

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            bs = x.size(0)

            seen += bs

            train_loss += (
                loss.item()
                *
                bs
            )

            cls_loss_sum += (
                loss_cls.item()
                *
                bs
            )

            dom_loss_sum += (
                loss_domain.item()
                *
                bs
            )

            con_loss_sum += (
                loss_supcon.item()
                *
                bs
            )

        scheduler.step()

        avg_train = (
            train_loss
            /
            max(
                1,
                seen
            )
        )

        avg_cls = (
            cls_loss_sum
            /
            max(
                1,
                seen
            )
        )

        avg_dom = (
            dom_loss_sum
            /
            max(
                1,
                seen
            )
        )

        avg_con = (
            con_loss_sum
            /
            max(
                1,
                seen
            )
        )

        val_metrics = evaluate_binary(
            model,
            val_loader,
            DEVICE
        )

        val_score = (
            validation_selection_score(
                val_metrics
            )
        )

        history.append({

            "epoch":
                epoch + 1,

            "train_loss":
                avg_train,

            "classification_loss":
                avg_cls,

            "domain_loss":
                avg_dom,

            "supcon_loss":
                avg_con,

            "domain_weight":
                domain_weight,

            "grl_lambda":
                grl_lambda,

            "learning_rate":
                optimizer.param_groups[0]["lr"],

            "val_accuracy":
                val_metrics["accuracy"],

            "val_balanced_accuracy":
                val_metrics[
                    "balanced_accuracy"
                ],

            "val_kappa":
                val_metrics["kappa"],

            "val_auc":
                val_metrics["roc_auc"],

            "val_score":
                val_score
        })

        # ----------------------------------------------------
        # Best validation model
        # ----------------------------------------------------

        if (
            epoch + 1 >= MIN_EPOCHS
            and
            val_score > best_score
        ):

            best_score = val_score

            best_epoch = (
                epoch + 1
            )

            best_state = {
                k:
                v.detach()
                .cpu()
                .clone()
                for k, v
                in model.state_dict().items()
            }

            patience_count = 0

        else:

            patience_count += 1

        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------

        if (
            epoch == 0
            or
            (epoch + 1) % 10 == 0
        ):

            print(
                f"Epoch {epoch+1:03d}/"
                f"{num_epochs} | "
                f"Loss={avg_train:.4f} | "
                f"Cls={avg_cls:.4f} | "
                f"Dom={avg_dom:.4f} | "
                f"ValAcc="
                f"{val_metrics['accuracy']*100:.2f}% | "
                f"ValBal="
                f"{val_metrics['balanced_accuracy']*100:.2f}% | "
                f"ValAUC="
                f"{val_metrics['roc_auc']:.4f}"
            )

        if (
            epoch + 1 >= MIN_EPOCHS
            and
            patience_count >= PATIENCE
        ):

            print(
                f"[EARLY STOP] epoch {epoch+1}"
            )

            break

    # --------------------------------------------------------
    # Restore best validation state
    # --------------------------------------------------------

    if best_state is None:

        # Fallback to final state
        best_state = {
            k:
            v.detach()
            .cpu()
            .clone()
            for k, v
            in model.state_dict().items()
        }

        best_epoch = num_epochs

    model.load_state_dict(
        best_state
    )

    history_df = pd.DataFrame(
        history
    )

    return (
        model,
        history_df,
        best_score,
        best_epoch
    )


print("[OK] Training function ready.")


In [ ]:
# ============================================================
# CELL 17 — OPTIONAL SOURCE-ONLY HYPERPARAMETER SCREEN
# ============================================================

def source_only_tune(
    train_ds,
    val_ds,
    fold_idx
):

    candidates = []

    print()
    print(
        "=" * 80
    )

    print(
        f"SOURCE-ONLY TUNING | "
        f"FOLD {fold_idx}"
    )

    print(
        "=" * 80
    )

    for candidate_idx, (
        candidate_lr,
        candidate_domain,
        candidate_supcon
    ) in enumerate(
        TUNE_CANDIDATES,
        start=1
    ):

        print()
        print(
            f"Candidate {candidate_idx}/"
            f"{len(TUNE_CANDIDATES)} | "
            f"LR={candidate_lr} | "
            f"Domain={candidate_domain} | "
            f"SupCon={candidate_supcon}"
        )

        model, hist, score, epoch = (
            train_model(
                train_ds,
                val_ds,
                fold_idx=
                    fold_idx * 100
                    +
                    candidate_idx,
                num_epochs=TUNE_EPOCHS,
                lr=candidate_lr,
                domain_weight_max=candidate_domain,
                supcon_weight=candidate_supcon
            )
        )

        val_loader = DataLoader(
            val_ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        metrics = evaluate_binary(
            model,
            val_loader,
            DEVICE
        )

        candidates.append({

            "candidate":
                candidate_idx,

            "learning_rate":
                candidate_lr,

            "domain_weight":
                candidate_domain,

            "supcon_weight":
                candidate_supcon,

            "val_accuracy":
                metrics["accuracy"],

            "val_balanced_accuracy":
                metrics[
                    "balanced_accuracy"
                ],

            "val_kappa":
                metrics["kappa"],

            "val_auc":
                metrics["roc_auc"],

            "selection_score":
                validation_selection_score(
                    metrics
                ),

            "best_epoch":
                epoch
        })

        del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    tune_df = pd.DataFrame(
        candidates
    )

    tune_df = tune_df.sort_values(
        "selection_score",
        ascending=False
    ).reset_index(
        drop=True
    )

    print()
    print(
        tune_df.to_string(
            index=False
        )
    )

    best = tune_df.iloc[
        0
    ]

    print()
    print(
        "Selected source-only configuration:"
    )

    print(
        best.to_dict()
    )

    return (
        float(best["learning_rate"]),
        float(best["domain_weight"]),
        float(best["supcon_weight"]),
        tune_df
    )


print(
    "[OK] Source-only tuning helper ready."
)


In [ ]:
# ============================================================
# CELL 18 — EXACT TEN-SUBJECT EXPERIMENT
# ============================================================

def run_exact_experiment():

    fold_rows = []

    prediction_rows = []

    history_rows = []

    tuning_rows = []

    embedding_rows = []

    experiment_start = time.time()

    for fold_idx, test_subject in enumerate(
        TEST_SUBJECTS,
        start=1
    ):

        print()
        print(
            "=" * 90
        )

        print(
            f"FOLD {fold_idx}/"
            f"{len(TEST_SUBJECTS)} | "
            f"TARGET S{test_subject:03d}"
        )

        print(
            "=" * 90
        )

        # ----------------------------------------------------
        # Source subjects
        # ----------------------------------------------------

        source_subjects = [
            s
            for s in available_subjects
            if s != test_subject
        ]

        # ----------------------------------------------------
        # Source-only validation split
        # ----------------------------------------------------

        train_subjects, val_subjects = (
            make_subject_validation_split(
                source_subjects,
                VAL_FRACTION,
                VAL_SEED + fold_idx
            )
        )

        # ----------------------------------------------------
        # Cached subsets
        # ----------------------------------------------------

        train_ds = EEG_CACHE.subset(
            train_subjects
        )

        val_ds = EEG_CACHE.subset(
            val_subjects
        )

        test_ds = EEG_CACHE.subset(
            [test_subject]
        )

        print(
            f"Subjects | "
            f"Train={len(train_subjects)} | "
            f"Val={len(val_subjects)} | "
            f"Test=1"
        )

        print(
            f"Trials   | "
            f"Train={len(train_ds)} | "
            f"Val={len(val_ds)} | "
            f"Test={len(test_ds)}"
        )

        # ----------------------------------------------------
        # Exact target class check
        # ----------------------------------------------------

        test_labels = np.asarray(
            test_ds.labels
        )

        if len(
            np.unique(
                test_labels
            )
        ) != 2:

            raise RuntimeError(
                f"S{test_subject:03d} "
                "does not contain both classes."
            )

        # ----------------------------------------------------
        # Optional source-only tuning
        # ----------------------------------------------------

        selected_lr = LR
        selected_domain = DOMAIN_WEIGHT_MAX
        selected_supcon = SUPCON_WEIGHT

        if RUN_SOURCE_TUNING:

            (
                selected_lr,
                selected_domain,
                selected_supcon,
                tune_df
            ) = source_only_tune(
                train_ds,
                val_ds,
                fold_idx
            )

            tune_df[
                "fold"
            ] = fold_idx

            tune_df[
                "test_subject"
            ] = test_subject

            tuning_rows.append(
                tune_df
            )

        else:

            print(
                "Source-only tuning disabled."
            )

        # ----------------------------------------------------
        # Final training
        # ----------------------------------------------------

        print()
        print(
            "FINAL TRAINING CONFIG:"
        )

        print(
            "  LR       :",
            selected_lr
        )

        print(
            "  Domain   :",
            selected_domain
        )

        print(
            "  SupCon   :",
            selected_supcon
        )

        model, history_df, best_score, best_epoch = (
            train_model(
                train_ds,
                val_ds,
                fold_idx,
                num_epochs=NUM_EPOCHS,
                lr=selected_lr,
                domain_weight_max=selected_domain,
                supcon_weight=selected_supcon
            )
        )

        history_df[
            "fold"
        ] = fold_idx

        history_df[
            "test_subject"
        ] = test_subject

        history_rows.append(
            history_df
        )

        # ----------------------------------------------------
        # Target loader
        # ----------------------------------------------------

        test_loader = DataLoader(
            test_ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        # ----------------------------------------------------
        # AdaBN — unlabeled target statistics
        # ----------------------------------------------------

        model = apply_adabn(
            model,
            test_loader,
            DEVICE,
            adaptation_trials=len(
                test_ds
            )
        )

        # ----------------------------------------------------
        # Final target evaluation
        # ----------------------------------------------------

        target_metrics = evaluate_binary(
            model,
            test_loader,
            DEVICE
        )

        cm = confusion_matrix(
            target_metrics["y_true"],
            target_metrics["y_pred"],
            labels=[0, 1]
        )

        print()
        print(
            f"S{test_subject:03d} FINAL"
        )

        print(
            f"Accuracy       : "
            f"{target_metrics['accuracy']*100:.2f}%"
        )

        print(
            f"Balanced Acc   : "
            f"{target_metrics['balanced_accuracy']*100:.2f}%"
        )

        print(
            f"Kappa          : "
            f"{target_metrics['kappa']:.4f}"
        )

        print(
            f"ROC-AUC        : "
            f"{target_metrics['roc_auc']:.4f}"
        )

        print(
            f"Best epoch     : "
            f"{best_epoch}"
        )

        print(
            "Confusion matrix:"
        )

        print(
            cm
        )

        # ----------------------------------------------------
        # Fold result
        # ----------------------------------------------------

        fold_rows.append({

            "fold":
                fold_idx,

            "test_subject":
                test_subject,

            "n_train_subjects":
                len(train_subjects),

            "n_val_subjects":
                len(val_subjects),

            "n_train_trials":
                len(train_ds),

            "n_val_trials":
                len(val_ds),

            "n_test_trials":
                len(test_ds),

            "accuracy":
                target_metrics[
                    "accuracy"
                ],

            "balanced_accuracy":
                target_metrics[
                    "balanced_accuracy"
                ],

            "kappa":
                target_metrics[
                    "kappa"
                ],

            "roc_auc":
                target_metrics[
                    "roc_auc"
                ],

            "precision_macro":
                target_metrics[
                    "precision_macro"
                ],

            "recall_macro":
                target_metrics[
                    "recall_macro"
                ],

            "f1_macro":
                target_metrics[
                    "f1_macro"
                ],

            "best_epoch":
                best_epoch,

            "best_val_score":
                best_score,

            "selected_lr":
                selected_lr,

            "selected_domain_weight":
                selected_domain,

            "selected_supcon_weight":
                selected_supcon
        })

        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        for i in range(
            len(
                target_metrics[
                    "y_true"
                ]
            )
        ):

            prediction_rows.append({

                "fold":
                    fold_idx,

                "test_subject":
                    test_subject,

                "true_label":
                    int(
                        target_metrics[
                            "y_true"
                        ][i]
                    ),

                "pred_label":
                    int(
                        target_metrics[
                            "y_pred"
                        ][i]
                    ),

                "prob_right":
                    float(
                        target_metrics[
                            "y_prob"
                        ][i]
                    ),

                "prob_left":
                    float(
                        1.0
                        -
                        target_metrics[
                            "y_prob"
                        ][i]
                    )
            })

            z = target_metrics[
                "embeddings"
            ][i]

            embedding_rows.append({

                "fold":
                    fold_idx,

                "test_subject":
                    test_subject,

                "true_label":
                    int(
                        target_metrics[
                            "y_true"
                        ][i]
                    ),

                **{
                    f"z_{j}":
                    float(z[j])

                    for j in range(
                        len(z)
                    )
                }
            })

        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del model
        del train_ds
        del val_ds
        del test_ds
        del test_loader

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ========================================================
    # BUILD DATAFRAMES
    # ========================================================

    fold_df = pd.DataFrame(
        fold_rows
    )

    prediction_df = pd.DataFrame(
        prediction_rows
    )

    history_df = pd.concat(
        history_rows,
        ignore_index=True
    )

    embedding_df = pd.DataFrame(
        embedding_rows
    )

    if tuning_rows:

        tuning_df = pd.concat(
            tuning_rows,
            ignore_index=True
        )

    else:

        tuning_df = pd.DataFrame()

    # ========================================================
    # POOLED METRICS
    # ========================================================

    pooled_accuracy = accuracy_score(
        prediction_df[
            "true_label"
        ],
        prediction_df[
            "pred_label"
        ]
    )

    pooled_balanced = balanced_accuracy_score(
        prediction_df[
            "true_label"
        ],
        prediction_df[
            "pred_label"
        ]
    )

    pooled_kappa = cohen_kappa_score(
        prediction_df[
            "true_label"
        ],
        prediction_df[
            "pred_label"
        ]
    )

    pooled_auc = roc_auc_score(
        prediction_df[
            "true_label"
        ],
        prediction_df[
            "prob_right"
        ]
    )

    # ========================================================
    # SUMMARY
    # ========================================================

    summary = {

        "task":
            "Binary Left Hand MI vs Right Hand MI",

        "runs":
            RUNS,

        "test_subjects":
            TEST_SUBJECTS,

        "completed_folds":
            int(len(fold_df)),

        "total_test_trials":
            int(
                prediction_df.shape[0]
            ),

        "pooled_accuracy":
            float(
                pooled_accuracy
            ),

        "pooled_balanced_accuracy":
            float(
                pooled_balanced
            ),

        "pooled_kappa":
            float(
                pooled_kappa
            ),

        "pooled_roc_auc":
            float(
                pooled_auc
            ),

        "mean_fold_accuracy":
            float(
                fold_df[
                    "accuracy"
                ].mean()
            ),

        "std_fold_accuracy":
            float(
                fold_df[
                    "accuracy"
                ].std(
                    ddof=0
                )
            ),

        "mean_balanced_accuracy":
            float(
                fold_df[
                    "balanced_accuracy"
                ].mean()
            ),

        "mean_kappa":
            float(
                fold_df[
                    "kappa"
                ].mean()
            ),

        "mean_roc_auc":
            float(
                fold_df[
                    "roc_auc"
                ].mean()
            ),

        "chance_accuracy":
            0.50,

        "device":
            str(DEVICE),

        "elapsed_minutes":
            float(
                (
                    time.time()
                    -
                    experiment_start
                )
                /
                60.0
            )
    }

    # ========================================================
    # SAVE
    # ========================================================

    fold_df.to_csv(
        RESULTS_DIR /
        "fold_metrics.csv",
        index=False
    )

    prediction_df.to_csv(
        RESULTS_DIR /
        "test_predictions.csv",
        index=False
    )

    history_df.to_csv(
        RESULTS_DIR /
        "training_history.csv",
        index=False
    )

    embedding_df.to_csv(
        RESULTS_DIR /
        "test_embeddings.csv",
        index=False
    )

    if not tuning_df.empty:

        tuning_df.to_csv(
            RESULTS_DIR /
            "source_only_tuning.csv",
            index=False
        )

    with open(
        RESULTS_DIR /
        "summary.json",
        "w"
    ) as f:

        json.dump(
            summary,
            f,
            indent=2
        )

    # ========================================================
    # PRINT FINAL
    # ========================================================

    print()
    print(
        "=" * 90
    )

    print(
        "FINAL BINARY LEFT-vs-RIGHT RESULTS"
    )

    print(
        "=" * 90
    )

    print(
        "Targets:",
        ", ".join(
            f"S{s:03d}"
            for s in TEST_SUBJECTS
        )
    )

    print()

    print(
        f"Pooled Accuracy       : "
        f"{pooled_accuracy*100:.2f}%"
    )

    print(
        f"Pooled Balanced Acc   : "
        f"{pooled_balanced*100:.2f}%"
    )

    print(
        f"Pooled Cohen's Kappa  : "
        f"{pooled_kappa:.4f}"
    )

    print(
        f"Pooled ROC-AUC        : "
        f"{pooled_auc:.4f}"
    )

    print()

    print(
        f"Mean Fold Accuracy    : "
        f"{summary['mean_fold_accuracy']*100:.2f}% "
        f"± "
        f"{summary['std_fold_accuracy']*100:.2f}%"
    )

    print(
        f"Mean Balanced Accuracy: "
        f"{summary['mean_balanced_accuracy']*100:.2f}%"
    )

    print(
        f"Mean Cohen's Kappa    : "
        f"{summary['mean_kappa']:.4f}"
    )

    print(
        f"Mean ROC-AUC          : "
        f"{summary['mean_roc_auc']:.4f}"
    )

    print(
        "Chance level          : 50.00%"
    )

    print()
    print(
        "PER-FOLD RESULTS"
    )

    print(
        fold_df[
            [
                "fold",
                "test_subject",
                "accuracy",
                "balanced_accuracy",
                "kappa",
                "roc_auc",
                "best_epoch"
            ]
        ].to_string(
            index=False
        )
    )

    print()
    print(
        "Saved to:",
        RESULTS_DIR.resolve()
    )

    return (
        fold_df,
        prediction_df,
        history_df,
        embedding_df,
        tuning_df,
        summary
    )


print(
    "[OK] Exact experiment function ready."
)


In [ ]:
# ============================================================
# CELL 19 — RUN THE EXPERIMENT
# ============================================================

RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:

    (
        fold_df,
        prediction_df,
        history_df,
        embedding_df,
        tuning_df,
        summary
    ) = run_exact_experiment()

else:

    print(
        "Full experiment is disabled."
    )

    print(
        "Set RUN_FULL_EXPERIMENT = True "
        "and run this cell to launch all 10 target folds."
    )


In [ ]:
# ============================================================
# CELL 20 — RESEARCH FIGURES
# ============================================================

def generate_figures(
    fold_df,
    prediction_df,
    history_df,
    embedding_df
):

    if (
        fold_df is None
        or
        fold_df.empty
    ):

        print(
            "No results available."
        )

        return

    # --------------------------------------------------------
    # FIGURE 1 — Fold accuracy
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(10, 5)
    )

    labels = [
        f"S{int(s):03d}"
        for s
        in fold_df[
            "test_subject"
        ]
    ]

    ax.bar(
        labels,
        fold_df[
            "accuracy"
        ]
        *
        100
    )

    ax.axhline(
        50,
        linestyle="--",
        linewidth=1.5,
        label="Chance"
    )

    ax.set_xlabel(
        "Held-out subject"
    )

    ax.set_ylabel(
        "Accuracy (%)"
    )

    ax.set_title(
        "Binary Subject-Independent Left-vs-Right Accuracy"
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        FIG_DIR /
        "Fig1_FoldAccuracy.png",
        dpi=400,
        bbox_inches="tight"
    )

    plt.close(fig)

    # --------------------------------------------------------
    # FIGURE 2 — Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        prediction_df[
            "true_label"
        ],
        prediction_df[
            "pred_label"
        ],
        labels=[0, 1]
    )

    cm_norm = (
        cm
        /
        np.maximum(
            cm.sum(
                axis=1,
                keepdims=True
            ),
            1
        )
    )

    fig, ax = plt.subplots(
        figsize=(6, 5)
    )

    im = ax.imshow(
        cm_norm,
        interpolation="nearest"
    )

    ax.set_title(
        "Normalized Binary Confusion Matrix"
    )

    ax.set_xlabel(
        "Predicted class"
    )

    ax.set_ylabel(
        "True class"
    )

    ax.set_xticks(
        [0, 1],
        ["Left", "Right"]
    )

    ax.set_yticks(
        [0, 1],
        ["Left", "Right"]
    )

    for i in range(2):

        for j in range(2):

            ax.text(
                j,
                i,
                f"{cm_norm[i,j]*100:.1f}%",
                ha="center",
                va="center"
            )

    fig.colorbar(
        im,
        ax=ax
    )

    fig.tight_layout()

    fig.savefig(
        FIG_DIR /
        "Fig2_ConfusionMatrix.png",
        dpi=400,
        bbox_inches="tight"
    )

    plt.close(fig)

    # --------------------------------------------------------
    # FIGURE 3 — Class metrics
    # --------------------------------------------------------

    report = classification_report(
        prediction_df[
            "true_label"
        ],
        prediction_df[
            "pred_label"
        ],
        labels=[0, 1],
        target_names=[
            "Left",
            "Right"
        ],
        output_dict=True,
        zero_division=0
    )

    precision = [
        report["Left"]["precision"] * 100,
        report["Right"]["precision"] * 100
    ]

    recall = [
        report["Left"]["recall"] * 100,
        report["Right"]["recall"] * 100
    ]

    f1 = [
        report["Left"]["f1-score"] * 100,
        report["Right"]["f1-score"] * 100
    ]

    x = np.arange(2)
    width = 0.25

    fig, ax = plt.subplots(
        figsize=(8, 5)
    )

    ax.bar(
        x - width,
        precision,
        width,
        label="Precision"
    )

    ax.bar(
        x,
        recall,
        width,
        label="Recall"
    )

    ax.bar(
        x + width,
        f1,
        width,
        label="F1"
    )

    ax.set_xticks(
        x,
        ["Left", "Right"]
    )

    ax.set_ylim(
        0,
        100
    )

    ax.set_ylabel(
        "Score (%)"
    )

    ax.set_title(
        "Binary Per-Class Metrics"
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        FIG_DIR /
        "Fig3_ClassMetrics.png",
        dpi=400,
        bbox_inches="tight"
    )

    plt.close(fig)

    # --------------------------------------------------------
    # FIGURE 4 — ROC
    # --------------------------------------------------------

    y_true = prediction_df[
        "true_label"
    ].to_numpy()

    y_prob = prediction_df[
        "prob_right"
    ].to_numpy()

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    fpr, tpr, _ = roc_curve(
        y_true,
        y_prob
    )

    roc_auc = auc(
        fpr,
        tpr
    )

    ax.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"AUC={roc_auc:.3f}"
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1
    )

    ax.set_xlabel(
        "False Positive Rate"
    )

    ax.set_ylabel(
        "True Positive Rate"
    )

    ax.set_title(
        "Binary ROC Curve"
    )

    ax.grid(
        True,
        alpha=0.25
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        FIG_DIR /
        "Fig4_ROC.png",
        dpi=400,
        bbox_inches="tight"
    )

    plt.close(fig)

    # --------------------------------------------------------
    # FIGURE 5 — Training curves
    # --------------------------------------------------------

    if (
        history_df is not None
        and
        not history_df.empty
    ):

        grouped = (
            history_df
            .groupby(
                "epoch"
            )
            .agg({

                "train_loss":
                    "mean",

                "classification_loss":
                    "mean",

                "domain_loss":
                    "mean",

                "supcon_loss":
                    "mean",

                "val_balanced_accuracy":
                    "mean"
            })
            .reset_index()
        )

        fig, ax = plt.subplots(
            figsize=(8, 5)
        )

        ax.plot(
            grouped["epoch"],
            grouped["train_loss"],
            label="Total loss",
            linewidth=2
        )

        ax.plot(
            grouped["epoch"],
            grouped["classification_loss"],
            label="Classification"
        )

        ax.plot(
            grouped["epoch"],
            grouped["domain_loss"],
            label="Domain"
        )

        ax.plot(
            grouped["epoch"],
            grouped["supcon_loss"],
            label="SupCon"
        )

        ax.set_xlabel(
            "Epoch"
        )

        ax.set_ylabel(
            "Loss"
        )

        ax.set_title(
            "Training Loss Components"
        )

        ax.grid(
            True,
            alpha=0.25
        )

        ax.legend()

        fig.tight_layout()

        fig.savefig(
            FIG_DIR /
            "Fig5_TrainingLoss.png",
            dpi=400,
            bbox_inches="tight"
        )

        plt.close(fig)

    # --------------------------------------------------------
    # FIGURE 6 — t-SNE
    # --------------------------------------------------------

    if (
        embedding_df is not None
        and
        len(embedding_df) >= 10
    ):

        z_cols = [
            c for c
            in embedding_df.columns
            if c.startswith("z_")
        ]

        if len(z_cols) >= 2:

            X = embedding_df[
                z_cols
            ].to_numpy()

            y = embedding_df[
                "true_label"
            ].to_numpy()

            perplexity = min(
                30,
                max(
                    5,
                    (
                        len(X) - 1
                    )
                    //
                    3
                )
            )

            tsne = TSNE(
                n_components=2,
                perplexity=perplexity,
                init="pca",
                learning_rate="auto",
                random_state=SEED
            )

            Z = tsne.fit_transform(
                X
            )

            fig, ax = plt.subplots(
                figsize=(8, 6)
            )

            for c, name in enumerate(
                CLASS_NAMES
            ):

                mask = (
                    y == c
                )

                ax.scatter(
                    Z[mask, 0],
                    Z[mask, 1],
                    s=12,
                    alpha=0.75,
                    label=name
                )

            ax.set_title(
                "t-SNE of Held-Out Target Embeddings"
            )

            ax.set_xlabel(
                "t-SNE dimension 1"
            )

            ax.set_ylabel(
                "t-SNE dimension 2"
            )

            ax.legend(
                fontsize=8
            )

            ax.grid(
                True,
                alpha=0.15
            )

            fig.tight_layout()

            fig.savefig(
                FIG_DIR /
                "Fig6_tSNE.png",
                dpi=400,
                bbox_inches="tight"
            )

            plt.close(fig)

    print(
        "Figures saved to:",
        FIG_DIR.resolve()
    )


if (
    "fold_df" in globals()
    and
    "prediction_df" in globals()
    and
    "history_df" in globals()
    and
    "embedding_df" in globals()
):

    generate_figures(
        fold_df,
        prediction_df,
        history_df,
        embedding_df
    )

else:

    print(
        "Run Cell 19 first."
    )


In [ ]:
# ============================================================
# CELL 21 — PAPER RESULTS EXPORT
# ============================================================

def export_paper_results(
    fold_df,
    summary
):

    if (
        fold_df is None
        or
        fold_df.empty
    ):

        print(
            "No results available."
        )

        return

    text = f"""
BINARY LEFT-vs-RIGHT MOTOR-IMAGERY RESULTS
==========================================

Completed folds: {len(fold_df)}

Test subjects:
{", ".join(f"S{s:03d}" for s in TEST_SUBJECTS)}

Pooled Accuracy:
{summary["pooled_accuracy"]*100:.2f}%

Pooled Balanced Accuracy:
{summary["pooled_balanced_accuracy"]*100:.2f}%

Pooled Cohen's Kappa:
{summary["pooled_kappa"]:.4f}

Pooled ROC-AUC:
{summary["pooled_roc_auc"]:.4f}

Mean Fold Accuracy:
{summary["mean_fold_accuracy"]*100:.2f}% ± {summary["std_fold_accuracy"]*100:.2f}%

Mean Balanced Accuracy:
{summary["mean_balanced_accuracy"]*100:.2f}%

Mean Cohen's Kappa:
{summary["mean_kappa"]:.4f}

Mean ROC-AUC:
{summary["mean_roc_auc"]:.4f}

Binary chance level:
50.00%

Per-fold:
{fold_df.to_string(index=False)}
""".strip()

    output = (
        RESULTS_DIR /
        "paper_results.txt"
    )

    output.write_text(
        text
    )

    print(
        text
    )

    print()
    print(
        "Saved:",
        output.resolve()
    )


if (
    "fold_df" in globals()
    and
    "summary" in globals()
):

    export_paper_results(
        fold_df,
        summary
    )

else:

    print(
        "Run Cell 19 first."
    )


## CELL 22 — Execution order

After **Kernel → Restart Kernel**:

`01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09 → 10 → 11 → 12 → 13 → 14 → 15 → 16 → 17 → 18 → 19 → 20 → 21`

For the first run, keep:

```python
RUN_SOURCE_TUNING = False
RUN_FULL_EXPERIMENT = False
```

Verify Cells 01–18 pass. Then set:

```python
RUN_FULL_EXPERIMENT = True
```

and run Cell 19.

### Optional source-only tuning

To enable the small source-validation hyperparameter screen:

```python
RUN_SOURCE_TUNING = True
```

This tuning **never uses the ten target subjects**. It selects among the predefined learning-rate/domain/SupCon candidates using source validation subjects only.

### Interpretation

Because each target subject has only a small number of trials, one correctly classified trial changes the fold accuracy by several percentage points. Report the ten-fold mean/SD together with pooled accuracy, balanced accuracy, κ, and ROC-AUC rather than treating one fold as definitive.

The notebook does not assume or guarantee a particular accuracy; it records the achieved result.
